# Performance estimation without current targets

This notebook shows two separate estimators. `DirectLossEstimator` learns regression loss from labeled reference predictions. `ConfidenceBasedPerformanceEstimator` uses calibrated class probabilities to estimate classification metrics. Their analyzers fit independently for each ID, and current targets are not required.


In [1]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd

from tinyshift.performance import DirectLossAnalyzer, DirectLossEstimator

rng = np.random.default_rng(42)

## 1. Build labeled reference and unlabeled current batches

The monitored model's prediction is `y_pred`. In this synthetic example, its absolute error grows with `risk`. The current batch for `store_A` has more high-risk observations; `store_B` has more low-risk observations. The `source` column is metadata, so we leave it out of `feature_cols`.


In [2]:
def make_batch(store, risk, *, labeled, source):
    y_pred = 10.0 + 0.5 * risk
    frame = pd.DataFrame({
        "unique_id": store,
        "risk": risk,
        "y_pred": y_pred,
        "source": source,
    })
    if labeled:
        expected_error = 0.3 + 1.5 * risk if store == "store_A" else 0.6 + 0.5 * risk
        frame["y"] = y_pred + expected_error + rng.normal(0, 0.04, len(risk))
    return frame


reference = pd.concat([
    make_batch("store_A", rng.uniform(0, 1, 400), labeled=True, source="reference"),
    make_batch("store_B", rng.uniform(0, 1, 400), labeled=True, source="reference"),
], ignore_index=True)

current = pd.concat([
    make_batch("store_A", rng.beta(8, 2, 120), labeled=False, source="current"),
    make_batch("store_B", rng.beta(2, 8, 120), labeled=False, source="current"),
], ignore_index=True)

print("Reference columns:", reference.columns.tolist())
print("Current columns:", current.columns.tolist())

Reference columns: ['unique_id', 'risk', 'y_pred', 'source', 'y']
Current columns: ['unique_id', 'risk', 'y_pred', 'source']


## 2. Fit one loss model per ID

Within each ID, the analyzer fits the loss model on the first 75% of reference rows and uses the final 25% as a held-out baseline. Keep each ID in the intended reference order; for time-series monitoring, sort by time before fitting. The estimator receives `risk` and the monitored model's `y_pred` internally.


In [3]:
analyzer = DirectLossAnalyzer(
    DirectLossEstimator(metric="mae"),
    validation_fraction=0.25,
).fit(reference, feature_cols=["risk"])

result = analyzer.predict(current)
result

  unique_id metric  ...  degradation  current_size
0   store_A    mae  ...         True           120
1   store_B    mae  ...        False           120

[2 rows x 9 columns]

## 3. Interpret the estimate

`reference_realized` is the observed MAE on held-out labeled reference rows. `reference_estimated` and `current_estimated` come from the same fitted loss model. `estimated_delta` compares those two estimates; `degradation` is simply whether that difference is positive.

The current frame has no `y`, so the example cannot verify its actual MAE. The result has **no p-value or hypothesis test**. An inaccurate loss model, especially after a large change in the population, can give an inaccurate performance estimate. Compare estimated and realized metrics when current labels eventually arrive.


In [4]:
print(result[[
    "unique_id",
    "reference_realized",
    "reference_estimated",
    "current_estimated",
    "estimated_delta",
    "degradation",
]].to_string(index=False))

unique_id  reference_realized  reference_estimated  current_estimated  estimated_delta  degradation
  store_A            1.089046             1.085171           1.505220         0.420049         True
  store_B            0.840542             0.857237           0.708394        -0.148843        False


## 4. Multiclass performance from probabilities

For each observation, the classifier supplies probabilities for **every class**. The estimator assigns the predicted class by the largest probability, then uses all class probabilities to build an expected confusion matrix. We generate reference labels from these probabilities, so the example is calibrated by construction. `store_A` becomes less confident in the current batch; `store_B` becomes more confident.


In [5]:
from tinyshift.performance import (
    ConfidenceBasedPerformanceAnalyzer,
    ConfidenceBasedPerformanceEstimator,
)

class_rng = np.random.default_rng(18)
classes = ["low", "medium", "high"]
probability_cols = dict(zip(classes, ["p_low", "p_medium", "p_high"]))


def make_class_batch(store, n, concentration, *, labeled):
    favored = class_rng.integers(0, len(classes), size=n)
    alpha = np.ones((n, len(classes)))
    alpha[np.arange(n), favored] = concentration
    probabilities = np.vstack([class_rng.dirichlet(row) for row in alpha])
    frame = pd.DataFrame(probabilities, columns=list(probability_cols.values()))
    frame.insert(0, "unique_id", store)
    if labeled:
        frame["y"] = [classes[class_rng.choice(len(classes), p=row)] for row in probabilities]
    return frame


classification_reference = pd.concat([
    make_class_batch("store_A", 400, 6, labeled=True),
    make_class_batch("store_B", 400, 6, labeled=True),
], ignore_index=True)
classification_current = pd.concat([
    make_class_batch("store_A", 150, 2, labeled=False),
    make_class_batch("store_B", 150, 12, labeled=False),
], ignore_index=True)

print("Reference columns:", classification_reference.columns.tolist())
print("Current columns:", classification_current.columns.tolist())

Reference columns: ['unique_id', 'p_low', 'p_medium', 'p_high', 'y']
Current columns: ['unique_id', 'p_low', 'p_medium', 'p_high']


## 5. Estimate accuracy by ID

`probability_cols` explicitly maps each class label to its probability column. The same interface works for binary classification with two columns. For precision, recall, or F1, pass that metric to `ConfidenceBasedPerformanceEstimator`; binary uses the final class as positive by default, while multiclass uses macro averaging.


In [6]:
classification_analyzer = ConfidenceBasedPerformanceAnalyzer(
    ConfidenceBasedPerformanceEstimator(metric="accuracy")
).fit(classification_reference, probability_cols=probability_cols)

classification_result = classification_analyzer.predict(classification_current)
print(classification_result.to_string(index=False))

unique_id   metric  reference_realized  reference_estimated  reference_size  current_estimated  estimated_delta  degradation  current_size
  store_A accuracy              0.7275             0.740519             400           0.597494        -0.143024         True           150
  store_B accuracy              0.7600             0.749954             400           0.865594         0.115640        False           150


`reference_realized` checks observed accuracy against the estimate on the labeled reference. `degradation` indicates a lower **estimated** metric in the current batch; there is no current `y` or hypothesis test. Reliable estimates require probabilities to remain calibrated. This implementation does not recalibrate probabilities automatically, so compare estimates with observed metrics when current labels arrive.


## 6. Binary probabilities and F1

For a binary classifier, many APIs return only `P(y=1)`. Derive the other column as `P(y=0) = 1 - P(y=1)`, then pass both columns in class order. The reference labels below are sampled from the predicted probabilities, making this synthetic example calibrated by construction. The current probabilities are closer to 0.5, so the expected F1 should fall.


In [7]:
binary_rng = np.random.default_rng(29)
reference_p1 = binary_rng.beta(2, 2, 500)
current_p1 = binary_rng.beta(8, 8, 150)

binary_reference = pd.DataFrame({
    "unique_id": "binary_model",
    "p0": 1 - reference_p1,
    "p1": reference_p1,
    "y": binary_rng.binomial(1, reference_p1),
})
binary_current = pd.DataFrame({
    "unique_id": "binary_model",
    "p0": 1 - current_p1,
    "p1": current_p1,
})

binary_analyzer = ConfidenceBasedPerformanceAnalyzer(
    ConfidenceBasedPerformanceEstimator(metric="f1", positive_label=1)
).fit(binary_reference, probability_cols={0: "p0", 1: "p1"})

print(binary_analyzer.predict(binary_current).to_string(index=False))

   unique_id metric  reference_realized  reference_estimated  reference_size  current_estimated  estimated_delta  degradation  current_size
binary_model     f1            0.665254             0.690526             500           0.612586        -0.077941         True           150


The expected confusion matrix has **predicted classes in rows** and **possible true classes in columns**. Each row of current probabilities contributes fractional expected counts. F1 is calculated from the aggregated matrix, rather than averaged per observation.


In [8]:
binary_estimator = binary_analyzer.estimators_["binary_model"]
expected_confusion = binary_estimator.expected_confusion_matrix(
    binary_current[["p0", "p1"]]
)
print(pd.DataFrame(
    expected_confusion,
    index=["predicted_0", "predicted_1"],
    columns=["true_0", "true_1"],
).round(2).to_string())

             true_0  true_1
predicted_0   42.63   28.37
predicted_1   31.59   47.41
